# 🇮🇹 Everyday Italian — Pimsleur-style audio lesson generator (B1)

**Lesson 2:** shopping · clothes & sales at the mall · grocery shopping · online shopping & returns · food delivery · restaurant & hotel reservations

Grammar in context: *ne* with quantities, *ce l'ha?*, combined pronouns (*me lo*, *glielo*), comparatives/superlatives, *conviene*, *dovrebbe*, *ci vuole*. Part 7 reviews sentences from Lesson 1.

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson02_Shopping_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# ITALIANO PER LA VITA QUOTIDIANA — Lesson 2 (B1)
# Theme: shopping — clothes at the mall, sales, groceries,
#        online shopping, food delivery, reservations
# Grammar: ne, direct/combined pronouns (ce l'ha, me lo), comparatives,
#          conditional, passato prossimo with pronouns, formal requests
#####################################################################

(section) Opening
(narrator) Everyday Italian. Lesson two. Shopping.
(narrator) Today you will learn to go shopping in Italian. You will buy clothes, food, and things online. You will also make a reservation.
(narrator) As before, answer out loud in the pause. Then repeat after the Italian speaker.
(narrator) Listen to this conversation. Giulia is in a clothing shop at the mall. It is the time of the winter sales.
(pause 1)
(it M) Buongiorno. Posso aiutarla?
(it F) Sì, grazie. Sto cercando una giacca per l'inverno.
(it M) Che taglia porta?
(it F) Porto la quarantadue. Questa giacca è in saldo?
(it M) Sì, c'è uno sconto del trenta per cento.
(it F) Posso provarla?
(it M) Certo. I camerini sono in fondo a destra.
(pause 1)
(it F) Mi sta un po' stretta. Ce l'ha in una taglia più grande?
(it M) Mi dispiace, la quarantaquattro è finita. Ma posso ordinarla online per Lei.
(it F) Quanto tempo ci vuole?
(it M) Arriva in tre o quattro giorni.
(it F) Va bene, la prendo. Posso pagare con la carta?
(it M) Certo. Vuole lo scontrino nella busta?
(it F) Sì, grazie. E se non mi va bene, posso cambiarla?
(it M) Sì, entro trenta giorni, con lo scontrino.
(pause 1.5)

(section) Part 1 — In the clothing shop
(narrator) Let's look at this conversation, piece by piece.
(narrator) The shop assistant says: can I help you? He uses the formal {Lei}, so the pronoun is {la}.
(repeat x2 slow) Posso aiutarla?
(narrator) Giulia says: I am looking for a jacket. {Sto cercando} means: I am looking for, right now.
(build) cando | cercando | sto cercando
(prompt) Say: I am looking for a jacket. || Sto cercando una giacca.
(prompt) Say: I am looking for a jacket for the winter. || Sto cercando una giacca per l'inverno.
(prompt) You are the shop assistant. Ask, formally: can I help you? || Posso aiutarla?
(narrator) Clothing size is {la taglia}. To ask the size, Italians use the verb {portare}.
(repeat x2 slow) Che taglia porta?
(prompt) Ask, formally: what size do you wear? || Che taglia porta?
(prompt) Answer: I wear a fifty. || Porto la cinquanta.
(narrator) The sales are {i saldi}. On sale is {in saldo}. A discount is {uno sconto}.
(repeat x2) i saldi
(prompt) Ask: is this jacket on sale? || Questa giacca è in saldo?
(prompt) Say: there is a thirty percent discount. || C'è uno sconto del trenta per cento.
(prompt) Say: the sales start in January. || I saldi cominciano a gennaio.
(prompt) Say: I am looking for a jacket for the winter. || Sto cercando una giacca per l'inverno.

(narrator) To try on is {provare}. Giulia says: can I try it on? The jacket is feminine, so {la}.
(repeat x2 slow) Posso provarla?
(narrator) For a sweater, {il maglione}, you say: {posso provarlo?}
(prompt) Ask: can I try it on? Talk about a sweater. || Posso provarlo?
(narrator) The fitting room is {il camerino}.
(build) rino | camerino | i camerini
(prompt) Say: the fitting rooms are at the back, on the right. || I camerini sono in fondo a destra.
(narrator) To say how clothes fit, use {stare}. {Mi sta stretta}: it is tight on me. {Mi sta larga}: it is loose on me.
(repeat x2 slow) Mi sta un po' stretta.
(prompt) Say: it is a bit tight on me. Talk about the jacket. || Mi sta un po' stretta.
(prompt) Say: it is too loose on me. Talk about the sweater. || Mi sta troppo largo.
(prompt) Say: it fits me well. Talk about the jacket. || Mi sta bene.
(prompt) Ask: is this jacket on sale? || Questa giacca è in saldo?

(narrator) Now a very useful question: do you have it? In Italian: {ce l'ha?}
(build) ce l'ha | ce l'ha?
(repeat x2 slow) Ce l'ha in una taglia più grande?
(prompt) Ask, formally: do you have it in a bigger size? || Ce l'ha in una taglia più grande?
(prompt) Ask: do you have it in a smaller size? || Ce l'ha in una taglia più piccola?
(prompt) Ask: do you have it in another colour? || Ce l'ha in un altro colore?
(prompt) Answer: I'm sorry, the forty-four is sold out. || Mi dispiace, la quarantaquattro è finita.
(narrator) To ask how long something takes, use {ci vuole}.
(prompt) Ask: how long does it take? || Quanto tempo ci vuole?
(prompt) Answer: it arrives in three or four days. || Arriva in tre o quattro giorni.
(prompt) Say: it is a bit tight on me. || Mi sta un po' stretta.

(narrator) To decide to buy, Italians say: I'll take it. {La prendo}, or {lo prendo}.
(prompt) Say: OK, I'll take it. Talk about the jacket. || Va bene, la prendo.
(prompt) Ask: can I pay by card? || Posso pagare con la carta?
(prompt) Ask: can I pay in cash? || Posso pagare in contanti?
(narrator) The receipt is {lo scontrino}. The bag is {la busta}, or {il sacchetto}.
(build) trino | scontrino | lo scontrino
(prompt) Ask, formally: do you want the receipt in the bag? || Vuole lo scontrino nella busta?
(prompt) Ask: if it doesn't fit me, can I exchange it? || Se non mi va bene, posso cambiarla?
(prompt) Answer: yes, within thirty days, with the receipt. || Sì, entro trenta giorni, con lo scontrino.
(narrator) If you want your money back, ask for {un rimborso}.
(prompt) Ask: can I have a refund? || Posso avere un rimborso?
(narrator) For a gift, you can ask: can you wrap it for me? {Me lo può incartare?}
(repeat x2 slow) Me lo può incartare? È un regalo.
(prompt) Ask, formally: can you wrap it for me? It's a gift. || Me lo può incartare? È un regalo.
(prompt) Ask: do you have it in a bigger size? || Ce l'ha in una taglia più grande?
(prompt) Ask: how long does it take? || Quanto tempo ci vuole?

(section) Part 2 — Grocery shopping
(narrator) Now let's buy food. To do the grocery shopping is {fare la spesa}.
(prompt) Say: I do the grocery shopping on Saturday morning. || Faccio la spesa il sabato mattina.
(prompt) Say: I usually go to the supermarket. || Di solito vado al supermercato.
(prompt) Say: but I buy fruit at the market. || Ma la frutta la compro al mercato.
(prompt) Say: we need to buy bread. Use {bisogna}. || Bisogna comprare il pane.
(prompt) Say: the milk is finished. || Il latte è finito.
(narrator) At the deli counter, Italians buy in {etti}. One {etto} is one hundred grams.
(build) etti | due etti | due etti di prosciutto
(prompt) Say: I would like two hundred grams of ham. || Vorrei due etti di prosciutto.
(prompt) Say: I would like half a kilo of cheese. || Vorrei mezzo chilo di formaggio.
(narrator) Now the little word {ne}. It means: of it, or of them. You use it with quantities.
(repeat x2 slow) Ne vorrei due etti.
(prompt) Say: I would like two hundred grams of it. || Ne vorrei due etti.
(prompt) Say: I would like a kilo of them. || Ne vorrei un chilo.
(prompt) Say: I would like three of them. || Ne vorrei tre.
(prompt) Say: I have already bought some. Use {ne}. || Ne ho già comprato.
(prompt) Ask: can I pay by card? || Posso pagare con la carta?

(prompt) Ask: how much are the peaches? || Quanto costano le pesche?
(prompt) Answer: they cost three euros per kilo. || Costano tre euro al chilo.
(prompt) Ask: are they fresh? || Sono fresche?
(narrator) The seller always asks: {altro?} Anything else? You can say: that's enough, thank you.
(repeat x2) Basta così, grazie.
(prompt) Say: nothing else, thank you. || Nient'altro, grazie.
(prompt) Say: I'd like a bag, please. || Vorrei un sacchetto, per favore.
(prompt) Say: I would like two hundred grams of it. || Ne vorrei due etti.

(narrator) Now a short role-play. You are at the market. Answer the seller.
(it M) Buongiorno! Che cosa desidera?
(prompt) Say: good morning. I would like a kilo of tomatoes. || Buongiorno. Vorrei un chilo di pomodori.
(it M) Ecco a Lei. Altro?
(prompt) Ask: how much are the peaches? || Quanto costano le pesche?
(it M) Due euro e cinquanta al chilo. Sono freschissime.
(prompt) Say: then I'd like half a kilo of them. || Allora ne vorrei mezzo chilo.
(it M) Benissimo. Altro?
(prompt) Say: that's enough, thank you. How much is it? || Basta così, grazie. Quant'è?
(it M) Sono quattro euro in tutto.
(prompt) Say: do you have change for twenty euros? || Ha il resto di venti euro?
(prompt) Say: if it doesn't fit me, can I exchange it? || Se non mi va bene, posso cambiarla?

(section) Part 3 — Prices, sales and comparisons
(narrator) When we shop, we compare. More expensive than is {più caro di}. Less expensive than is {meno caro di}.
(prompt) Say: this one is more expensive than that one. || Questo è più caro di quello.
(prompt) Say: it is less expensive online. || Online è meno caro.
(prompt) Say: this shop is cheaper. Use {economico}. || Questo negozio è più economico.
(prompt) Say: it is the cheapest shop in the city. || È il negozio più economico della città.
(narrator) A bargain is {un affare}.
(prompt) Say: what a bargain! || Che affare!
(prompt) Say: I spent too much. || Ho speso troppo.
(prompt) Say: during the sales, I always spend too much. || Durante i saldi spendo sempre troppo.
(narrator) {Conviene} means: it is worth it, it is better value.
(prompt) Say: it is better to buy online. Use {conviene}. || Conviene comprare online.
(prompt) Ask: is it worth waiting for the sales? || Conviene aspettare i saldi?
(prompt) Say: when I was a student, I never bought new clothes. || Quando ero studente, non compravo mai vestiti nuovi.
(prompt) Say: there is a thirty percent discount. || C'è uno sconto del trenta per cento.
(prompt) Say: I would like half a kilo of cheese. || Vorrei mezzo chilo di formaggio.

(section) Part 4 — Online shopping
(narrator) Now online shopping. To order is {ordinare}. A package is {il pacco}. Shipping is {la spedizione}.
(prompt) Say: I ordered a pair of shoes online. || Ho ordinato un paio di scarpe online.
(prompt) Say: I added them to the cart. || Le ho aggiunte al carrello.
(prompt) Say: shipping is free. || La spedizione è gratuita.
(build) zione | dizione | spedizione
(prompt) Say: the package hasn't arrived yet. || Il pacco non è ancora arrivato.
(narrator) To say what should happen, use the conditional of {dovere}: {dovrebbe}.
(prompt) Say: it should arrive tomorrow. || Dovrebbe arrivare domani.
(prompt) Ask: do you have the tracking code? || Hai il codice di tracciamento?
(prompt) Say: the payment didn't go through. || Il pagamento non è andato a buon fine.
(prompt) Say: this one is more expensive than that one. || Questo è più caro di quello.

(narrator) To send something back is {restituire}. A return, online, is {il reso}.
(build) tuire | restituire
(prompt) Say: I would like to return this product. || Vorrei restituire questo prodotto.
(prompt) Say: the shoes are too small. || Le scarpe sono troppo piccole.
(prompt) Ask: is the return free? || Il reso è gratuito?
(prompt) Say: they refunded me after a week. || Mi hanno rimborsato dopo una settimana.
(narrator) Now a combined pronoun. I send it to him, or to her, or to you formally: {glielo spedisco}.
(repeat x2 slow) Glielo spedisco domani.
(prompt) You are the seller. Say, formally: I will send it to you tomorrow. || Glielo spedisco domani.
(prompt) Say: it should arrive tomorrow. || Dovrebbe arrivare domani.
(prompt) Say: I do the grocery shopping on Saturday morning. || Faccio la spesa il sabato mattina.

(section) Part 5 — Ordering food
(narrator) Now let's order food. Home delivery is {la consegna a domicilio}.
(build) cilio | domicilio | a domicilio
(prompt) Ask a friend: shall we order a pizza? Use "a domicilio". || Ordiniamo una pizza a domicilio?
(prompt) Say: I ordered with an app. || Ho ordinato con un'app.
(prompt) Say: a margherita and a diavola, please. || Una margherita e una diavola, per favore.
(prompt) Ask: how long does it take? || Quanto ci vuole?
(prompt) Answer: about half an hour. || Circa mezz'ora.
(narrator) The delivery person is {il fattorino}, or {il rider}.
(prompt) Say: the delivery person is late. || Il fattorino è in ritardo.
(prompt) Say: the pizza arrived cold. || La pizza è arrivata fredda.
(prompt) Say: I would like to return this product. || Vorrei restituire questo prodotto.
(prompt) Say: what a bargain! || Che affare!

(section) Part 6 — Making reservations
(narrator) Finally, reservations. To book is {prenotare}. A reservation is {una prenotazione}.
(build) tare | prenotare
(build) zione | tazione | prenotazione
(prompt) Say: I would like to book a table. || Vorrei prenotare un tavolo.
(prompt) Say: I would like to book a table for four people. || Vorrei prenotare un tavolo per quattro persone.
(prompt) Now add: for Saturday evening. Say the whole sentence. || Vorrei prenotare un tavolo per quattro persone, per sabato sera.
(narrator) The restaurant asks: under what name? {A che nome?}
(prompt) Answer: under the name Rossi. || A nome Rossi.
(prompt) The restaurant asks: for what time? || Per che ora?
(prompt) Answer: at eight thirty. || Alle otto e mezza.
(prompt) Ask: do you have a table outside? || Avete un tavolo all'aperto?
(prompt) Answer: I'm sorry, we are fully booked. || Mi dispiace, siamo al completo.
(prompt) Say: I would like to cancel the reservation. || Vorrei cancellare la prenotazione.
(narrator) Now a hotel. A double room is {una camera doppia}.
(prompt) Say: I would like to book a double room for two nights. || Vorrei prenotare una camera doppia per due notti.
(prompt) Ask: is breakfast included? || La colazione è inclusa?
(prompt) Ask: how much does it cost per night? || Quanto costa a notte?
(prompt) Say: I ordered a pair of shoes online. || Ho ordinato un paio di scarpe online.

(narrator) A role-play. You call a restaurant. Answer the waiter.
(it M) Ristorante Da Mario, buonasera.
(prompt) Say: good evening. I would like to book a table for Saturday evening. || Buonasera. Vorrei prenotare un tavolo per sabato sera.
(it M) Certo. Per quante persone?
(prompt) Say: for four people. Do you have a table outside? || Per quattro persone. Avete un tavolo all'aperto?
(it M) Sì, abbiamo un tavolo in terrazza. Per che ora?
(prompt) Say: at eight thirty, if possible. || Alle otto e mezza, se possibile.
(it M) Perfetto. A che nome?
(prompt) Say: under the name Rossi. Thank you very much. || A nome Rossi. Grazie mille.

(section) Part 7 — Review
(narrator) Let's review. First, some sentences from lesson one.
(prompt) Say: I would like to register for your course, but the deadline has already passed. || Vorrei iscrivermi al suo corso, ma la scadenza è già passata.
(prompt) Say: they told me that your signature is needed. || Mi hanno detto che serve la sua firma.
(prompt) Ask, formally: could you explain this exercise again? || Potrebbe spiegarmi di nuovo questo esercizio?
(prompt) Say: I think the exam is difficult. || Penso che l'esame sia difficile.
(prompt) Say: I have lunch at the cafeteria with my classmates. || Pranzo in mensa con i miei compagni di corso.
(narrator) Now today's lesson. Answer quickly.
(prompt) Say: I am looking for a jacket for the winter. || Sto cercando una giacca per l'inverno.
(prompt) Ask: do you have it in a bigger size? || Ce l'ha in una taglia più grande?
(prompt) Say: it is a bit tight on me. || Mi sta un po' stretta.
(prompt) Ask: if it doesn't fit me, can I exchange it? || Se non mi va bene, posso cambiarla?
(prompt) Say: I would like two hundred grams of it. || Ne vorrei due etti.
(prompt) Say: it is better to buy online. || Conviene comprare online.
(prompt) Say: the package hasn't arrived yet. || Il pacco non è ancora arrivato.
(prompt) Say: I would like to return this product. || Vorrei restituire questo prodotto.
(prompt) Ask a friend: shall we order a pizza? || Ordiniamo una pizza a domicilio?
(prompt) Say: I would like to book a table for four people, for Saturday evening. || Vorrei prenotare un tavolo per quattro persone, per sabato sera.

(narrator) Now you are Giulia. Answer the shop assistant.
(it M) Buongiorno. Posso aiutarla?
(prompt F) Say: yes, thank you. I am looking for a jacket for the winter. || Sì, grazie. Sto cercando una giacca per l'inverno.
(it M) Che taglia porta?
(prompt F) Say: I wear a forty-two. Is this jacket on sale? || Porto la quarantadue. Questa giacca è in saldo?
(it M) Sì, c'è uno sconto del trenta per cento. Vuole provarla?
(prompt F) Say: yes. Hmm, it is a bit tight on me. Do you have it in a bigger size? || Sì. Mi sta un po' stretta. Ce l'ha in una taglia più grande?
(it M) Posso ordinarla online per Lei. Arriva in tre giorni.
(prompt F) Say: OK, I'll take it. Can I pay by card? || Va bene, la prendo. Posso pagare con la carta?

(narrator) Now listen to the first conversation again.
(pause 1)
(it M) Buongiorno. Posso aiutarla?
(it F) Sì, grazie. Sto cercando una giacca per l'inverno.
(it M) Che taglia porta?
(it F) Porto la quarantadue. Questa giacca è in saldo?
(it M) Sì, c'è uno sconto del trenta per cento.
(it F) Posso provarla?
(it M) Certo. I camerini sono in fondo a destra.
(pause 1)
(it F) Mi sta un po' stretta. Ce l'ha in una taglia più grande?
(it M) Mi dispiace, la quarantaquattro è finita. Ma posso ordinarla online per Lei.
(it F) Quanto tempo ci vuole?
(it M) Arriva in tre o quattro giorni.
(it F) Va bene, la prendo. Posso pagare con la carta?
(it M) Certo. Vuole lo scontrino nella busta?
(it F) Sì, grazie. E se non mi va bene, posso cambiarla?
(it M) Sì, entro trenta giorni, con lo scontrino.
(pause 1.5)
(narrator) This is the end of lesson two. Listen to it again tomorrow. {Buono shopping!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
